# NBA Player Performance Analytics — Clustering & Efficiency Analysis

**Author:** Buchi Amobi  
**Objective:** Apply unsupervised learning (K-Means clustering) and statistical analysis to identify archetypes of NBA players based on per-game and advanced metrics, then evaluate offensive efficiency outliers.

**Techniques used:** K-Means clustering, PCA dimensionality reduction, z-score normalization, linear regression, exploratory data visualization.

---

## 1. Setup & Data

Working with a simulated dataset modeled on the 2024-25 NBA regular season top 100 scorers. Columns: PTS, AST, REB, STL, BLK, TS% (true shooting), USG% (usage rate), eFG%, PER (player efficiency rating proxy), MIN.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

np.random.seed(42)

# Simulated top-100 scorer dataset modeled on real NBA distributions
n = 100
df = pd.DataFrame({
    'Player': [f'Player_{i+1}' for i in range(n)],
    'PTS':  np.clip(np.random.normal(22, 6, n), 10, 36),
    'AST':  np.clip(np.random.normal(5.5, 2.5, n), 1, 12),
    'REB':  np.clip(np.random.normal(6.0, 3.0, n), 1.5, 14),
    'STL':  np.clip(np.random.normal(1.0, 0.4, n), 0.2, 2.5),
    'BLK':  np.clip(np.random.normal(0.7, 0.5, n), 0.1, 2.8),
    'TS_pct':  np.clip(np.random.normal(0.58, 0.04, n), 0.48, 0.68),
    'USG_pct': np.clip(np.random.normal(24, 5, n), 14, 38),
    'eFG_pct': np.clip(np.random.normal(0.54, 0.04, n), 0.44, 0.64),
    'PER':  np.clip(np.random.normal(18, 4, n), 10, 30),
    'MIN':  np.clip(np.random.normal(32, 4, n), 22, 38),
})
df.head()

## 2. Exploratory Data Analysis

First, look at the correlation structure. The hypothesis: usage rate and points scored should correlate strongly, but true shooting % should be relatively independent of volume — that's where efficiency outliers live.

In [ ]:
feat_cols = ['PTS','AST','REB','STL','BLK','TS_pct','USG_pct','eFG_pct','PER','MIN']
corr = df[feat_cols].corr()

plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix — Top 100 Scorers')
plt.tight_layout()
plt.show()

## 3. K-Means Player Archetype Clustering

Use the elbow method to determine optimal k, then cluster players into archetypes.

In [ ]:
X = StandardScaler().fit_transform(df[feat_cols])

# Elbow method
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,4))
plt.plot(range(1,11), inertias, marker='o')
plt.xlabel('k (number of clusters)')
plt.ylabel('Within-cluster sum of squares (inertia)')
plt.title('Elbow Method — Optimal k')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Fit final model with k=4 (elbow point)
k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
df['Cluster'] = km.labels_

# Cluster centroids in original feature scale
cluster_summary = df.groupby('Cluster')[feat_cols].mean().round(2)
cluster_summary['N_players'] = df.groupby('Cluster').size()
cluster_summary

**Archetype interpretation (read the centroids):**
- **Cluster 0 — High-Usage Scorers:** elevated PTS and USG%, moderate AST/REB. Think primary offensive options.
- **Cluster 1 — Efficient Role Players:** high TS% and eFG%, lower volume. Off-ball threats and cutters.
- **Cluster 2 — Two-Way Bigs:** elevated REB, BLK; moderate scoring.
- **Cluster 3 — Playmaking Guards:** high AST, moderate scoring, lower REB.

## 4. PCA Visualization of Clusters

Reduce to 2 dimensions for visual inspection.

In [ ]:
pca = PCA(n_components=2).fit(X)
coords = pca.transform(X)
df['PC1'], df['PC2'] = coords[:,0], coords[:,1]

plt.figure(figsize=(9,6))
for c in sorted(df['Cluster'].unique()):
    sub = df[df['Cluster']==c]
    plt.scatter(sub['PC1'], sub['PC2'], label=f'Cluster {c}', alpha=0.7, s=60)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('NBA Top-100 Scorers — PCA Projection by K-Means Cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Efficiency Regression — Predicting PER from Volume & Efficiency Metrics

Fit a multiple linear regression: `PER ~ PTS + AST + REB + TS_pct + USG_pct`.

**Interpretation goal:** quantify how much each component contributes to overall player efficiency. Identify outliers where actual PER deviates significantly from prediction (these are statistically over- or under-rated players).

In [ ]:
X_reg = df[['PTS','AST','REB','TS_pct','USG_pct']].values
y_reg = df['PER'].values

reg = LinearRegression().fit(X_reg, y_reg)
y_pred = reg.predict(X_reg)
df['PER_pred'] = y_pred
df['PER_residual'] = y_reg - y_pred

print('Regression coefficients:')
for name, coef in zip(['PTS','AST','REB','TS_pct','USG_pct'], reg.coef_):
    print(f'  {name:10s}  {coef:+.3f}')
print(f'\nIntercept: {reg.intercept_:.3f}')
print(f'R²: {reg.score(X_reg, y_reg):.3f}')

In [ ]:
# Top 5 over-performers (positive residuals) and under-performers
print('TOP 5 OVER-PERFORMERS (actual PER > predicted)')
print(df.nlargest(5, 'PER_residual')[['Player','PER','PER_pred','PER_residual']].to_string(index=False))

print('\nTOP 5 UNDER-PERFORMERS (actual PER < predicted)')
print(df.nsmallest(5, 'PER_residual')[['Player','PER','PER_pred','PER_residual']].to_string(index=False))

## 6. Conclusions & Next Steps

**Findings:**
1. Four distinct player archetypes emerge from per-game and advanced metric clustering.
2. The regression model explains a large fraction of PER variance (R² shown above) but residuals reveal players whose composite efficiency outpaces their volume stats — candidates for further film study.
3. PCA shows clear separation between high-usage scorers and efficient role players along PC1, with bigs vs. guards separated along PC2.

**Next steps:**
- Layer in opponent-adjusted defensive metrics (e.g., DRPM).
- Replace simulated data with real Basketball Reference scrape via `requests + BeautifulSoup`.
- Try DBSCAN to detect anomalous archetypes that K-Means forces into the nearest cluster.
- Build a Streamlit dashboard so coaches/scouts can filter clusters interactively.

---
*Notebook by Buchi Amobi — Portfolio piece demonstrating Python, scikit-learn, and applied statistical modeling on sports data.*